# TODO 6.2：AWQ INT4 实验

本 Notebook 使用真实 Qwen 模型和 `calibration_finqa_1024`，测试 AWQ W4A16 的 group size 32、64、128。模型与数据仍在 `banking_llm_project`，只有量化模型和证据写入 `OptimizationSystem/out/awq`。

请先完成 4.1、4.2，并使用 T4 GPU。AutoAWQ 安装后通常需要重启 Colab 运行时，再从头运行本 Notebook。

In [ ]:
%pip -q install autoawq
# 安装后若提示需要重启运行时，请重启，再运行后续单元。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib
import json
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CONFIG_PATH = PROJECT_DIR / 'configs/awq.json'
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
RESULT_DIR = PROJECT_DIR / 'out/awq'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR))
import src.data.finqa_assets as finqa_assets
import src.quantization.awq as awq
importlib.reload(awq)
assets = finqa_assets.load_assets(DATASET_DIR)
print('校准样本:', len(assets.calibration))
print('结果目录:', RESULT_DIR)

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plans = awq.build_awq_plans(config)
print([plan.to_dict() for plan in plans])
assert MODEL_MANIFEST.is_file(), f'找不到模型 manifest: {MODEL_MANIFEST}'
assert CALIBRATION_MANIFEST.is_file(), f'找不到校准 manifest: {CALIBRATION_MANIFEST}'
print('AWQ 配置和输入资产：PASS')

## 真实执行
下面单元格会依次执行三个 group size。每个计划都可能占用数分钟和较多显存；不要把输出目录改到模型目录。

In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError('请切换到 NVIDIA T4 GPU。')
execution_results = []
for plan in plans:
    plan_dir = RESULT_DIR / plan.plan_id
    print(f'开始 AWQ: {plan.plan_id}')
    execution_results.append(awq.run_awq_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=plan_dir,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
    ))
    del plan_dir
    gc.collect()
    torch.cuda.empty_cache()
print(json.dumps(execution_results, ensure_ascii=False, indent=2))

In [ ]:
evidence = awq.collect_awq_evidence(
    config_file=CONFIG_PATH,
    calibration_manifest_file=CALIBRATION_MANIFEST,
    model_manifest_file=MODEL_MANIFEST,
    output_dir=RESULT_DIR,
)
evidence_path = RESULT_DIR / 'awq_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = awq.validate_awq_evidence(evidence_path)
validation_path = RESULT_DIR / 'validation.json'
validation_path.write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))
assert report.complete, 'AWQ 三个 group size 未全部完成，请查看 validation.json。'
print('TODO 6.2 AWQ 导出验收：PASS')
print('结果目录:', RESULT_DIR)